# 03 — BM25 + hybrid + metadata filters

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import DOCS_PARQUET
from src.retrieval.bm25 import BM25Retriever
from src.retrieval.faiss_index import FaissRetriever
from src.retrieval.hybrid import HybridRetriever

docs = pd.read_parquet(DOCS_PARQUET)
bm25 = BM25Retriever.build(docs["document"].tolist(), docs["business_id"].astype(str).tolist())
bm25.save()
engine = HybridRetriever(docs, FaissRetriever.load(), bm25)
q = "Find a quiet Italian restaurant for a date with vegetarian options"
for h in engine.search(q, method="hybrid", top_k=5):
    print(round(h.score, 3), h.metadata.get("name"), h.metadata.get("quiet"), h.metadata.get("vegetarian_friendly"))